# Trurning angle calculation from smooth csv

In [2]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Save csv for truning angle calculated from smooth data

In [3]:
# Input
INPUT_DIR  = r"D:\Nishka_Analysis\Analysis_Body_Wing\Data_DLC_DlTdv8\DLTdv8_Data\Final_DLTdv87_Data\Four_Chase_CSVData\xyz_Smooth"
OUTPUT_DIR = r"D:\Nishka_Analysis\Analysis_Body_Wing\Data_DLC_DlTdv8\DLTdv8_Data\Final_DLTdv87_Data\Four_Chase_CSVData\Turning_angle"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FPS = 2500   # frame rate


def normalize(v):
    n = np.linalg.norm(v, axis=1, keepdims=True)
    n[n == 0] = 1
    return v / n

# Find Smooth files
csv_files = glob.glob(os.path.join(INPUT_DIR, "*_SMOOTH.csv"))
if not csv_files:
    raise FileNotFoundError("No SMOOTH CSV files found.")


for path in csv_files:
    fname = os.path.basename(path)
    print("Processing:", fname)

    df = pd.read_csv(path)

    frames = df["frame"].values

    head   = df[["pt2_X","pt2_Y","pt2_Z"]].values
    center = df[["center_X","center_Y","center_Z"]].values

  
    # Heading vector (CENTER → HEAD)
    heading = normalize(head - center)

    # Turning angle between consecutive headings
    v1 = heading[:-1]
    v2 = heading[1:]

    dot = np.einsum("ij,ij->i", v1, v2)
    dot = np.clip(dot, -1, 1)

    turn_angle = np.degrees(np.arccos(dot))
    turn_angle_unwrapped = np.unwrap(np.radians(turn_angle)) * 180 / np.pi

    # Time axis (seconds)
    frame_mid = frames[1:]          # aligns with angle
    time_sec = frame_mid / FPS

    # Save csv
    out_df = pd.DataFrame({
        "frame": frame_mid,
        "time_sec": time_sec,
        "turning_angle_deg": turn_angle_unwrapped
    })

    out_name = fname.replace("_SMOOTH.csv", "_TURNING_ANGLE.csv")
    out_df.to_csv(os.path.join(OUTPUT_DIR, out_name), index=False)

    print("Saved:", out_name)

print("\n Turning angle CSVs generated.")


Processing: Final_DLTdv87_Data_DLTdv8_data_Trial2_200rpmxyzpts_SMOOTH.csv
Saved: Final_DLTdv87_Data_DLTdv8_data_Trial2_200rpmxyzpts_TURNING_ANGLE.csv
Processing: Final_DLTdv87_Data_DLTdv8_data_Trial3_180rpmxyzpts_SMOOTH.csv
Saved: Final_DLTdv87_Data_DLTdv8_data_Trial3_180rpmxyzpts_TURNING_ANGLE.csv
Processing: Final_DLTdv87_Data_DLTdv8_data_Trial4_400rpmxyzpts_SMOOTH.csv
Saved: Final_DLTdv87_Data_DLTdv8_data_Trial4_400rpmxyzpts_TURNING_ANGLE.csv
Processing: Final_DLTdv87_Data_DLTdv8_data_Trial5_180rpmxyzpts_SMOOTH.csv
Saved: Final_DLTdv87_Data_DLTdv8_data_Trial5_180rpmxyzpts_TURNING_ANGLE.csv

 Turning angle CSVs generated.


# Display and save plot of truning angle

In [4]:
# Where TURNING_ANGLE CSVs live
INPUT_DIR = r"D:\Nishka_Analysis\Analysis_Body_Wing\Data_DLC_DlTdv8\DLTdv8_Data\Final_DLTdv87_Data\Four_Chase_CSVData\Turning_angle"

# Base directory containing Trial2_200rpm, Trial3_180rpm, etc.
BASE_TRIAL_DIR = r"D:\Nishka_Analysis\Analysis_Body_Wing\Four_Chase_Data"

# ==========================================================
# FIND TURNING ANGLE FILES
# ==========================================================

csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*_TURNING_ANGLE.csv")))
if not csv_files:
    raise FileNotFoundError("No TURNING_ANGLE CSV files found.")

# ==========================================================
# PLOTTING
# ==========================================================

for path in csv_files:
    base = os.path.splitext(os.path.basename(path))[0]
    print("Plotting:", base)

    # ------------------------------------------------------
    # Extract trial name (e.g. Trial2_200rpm)
    # ------------------------------------------------------
    try:
        trial_name = [p for p in base.split("_") if p.startswith("Trial")][0]
    except IndexError:
        raise ValueError(f"Could not extract trial name from: {base}")

    # ------------------------------------------------------
    # Create trial-specific TURNING_ANGLE_PLOTS folder
    # ------------------------------------------------------
    trial_dir = os.path.join(BASE_TRIAL_DIR, trial_name)
    save_dir = os.path.join(trial_dir, "TURNING_ANGLE_PLOTS")
    os.makedirs(save_dir, exist_ok=True)

    # ------------------------------------------------------
    # Load data
    # ------------------------------------------------------
    df = pd.read_csv(path)

    # ------------------------------------------------------
    # Plot
    # ------------------------------------------------------
    plt.figure(figsize=(14, 6))
    plt.plot(
        df["time_sec"],
        df["turning_angle_deg"],
        color="darkgreen",
        linewidth=2
    )

    plt.xlabel("Time (seconds)")
    plt.ylabel("Turning Angle (°)")
    plt.title("Turning Angle vs Time")
    plt.grid(True)
    plt.tight_layout()

    out_path = os.path.join(save_dir, f"{base}.png")
    plt.savefig(out_path, dpi=300)
    plt.close()

    print("Saved:", out_path)

print("\n Turning angle plots saved trial-wise.")


Plotting: Final_DLTdv87_Data_DLTdv8_data_Trial2_200rpmxyzpts_TURNING_ANGLE
Saved: D:\Nishka_Analysis\Analysis_Body_Wing\Four_Chase_Data\Trial2\TURNING_ANGLE_PLOTS\Final_DLTdv87_Data_DLTdv8_data_Trial2_200rpmxyzpts_TURNING_ANGLE.png
Plotting: Final_DLTdv87_Data_DLTdv8_data_Trial3_180rpmxyzpts_TURNING_ANGLE
Saved: D:\Nishka_Analysis\Analysis_Body_Wing\Four_Chase_Data\Trial3\TURNING_ANGLE_PLOTS\Final_DLTdv87_Data_DLTdv8_data_Trial3_180rpmxyzpts_TURNING_ANGLE.png
Plotting: Final_DLTdv87_Data_DLTdv8_data_Trial4_400rpmxyzpts_TURNING_ANGLE
Saved: D:\Nishka_Analysis\Analysis_Body_Wing\Four_Chase_Data\Trial4\TURNING_ANGLE_PLOTS\Final_DLTdv87_Data_DLTdv8_data_Trial4_400rpmxyzpts_TURNING_ANGLE.png
Plotting: Final_DLTdv87_Data_DLTdv8_data_Trial5_180rpmxyzpts_TURNING_ANGLE
Saved: D:\Nishka_Analysis\Analysis_Body_Wing\Four_Chase_Data\Trial5\TURNING_ANGLE_PLOTS\Final_DLTdv87_Data_DLTdv8_data_Trial5_180rpmxyzpts_TURNING_ANGLE.png

✅ Turning angle plots saved trial-wise.
